# Hello World example - trustify Python package

This notebook walks through the public Python API of `trustify`: generating a schema from your TRUST sources, parsing a `.data` file into a pydantic model, inspecting and editing it, and writing it back out.

All snippets below import from the top-level `trustify` package — nothing under `trustify.core.*`, which is internal.

To run this notebook properly, source your TRUST environment first so that `$TRUST_ROOT` is set and points at your TRUST checkout:

```
    source $TRUST_ROOT/env_for_python.sh
```


## Generating the schema

`trustify.generate_schema()` reads the `// XD` declarations from your TRUST sources and produces a pydantic model + parser describing every keyword. With no argument it picks up the `TRUST_ROOT` environment variable. The output lives in a content-hashed directory under `~/.cache/trustify/`, so it is generated once and reused across runs (and across notebooks).

You usually do not need to call this explicitly — `load_dataset` below will trigger generation on demand — but it is the right entry point if you want the cache warmed up, or if you need the path to feed into another tool.

In [ ]:
import trustify

schema_dir = trustify.generate_schema()
print("Schema cache directory:", schema_dir)

`generate_schema` accepts:

- `trust_root` *(str, optional)* — path to your TRUST checkout. Falls back to the `TRUST_ROOT` environment variable when omitted.
- `projects` *(list[str], optional)* — additional project directories (typically BALTIK source trees) to overlay on top of TRUST. Each must contain a `src/` subdirectory with C++ / XD sources. Later entries override earlier ones on filename collision.
- `out` *(str | Path, optional)* — write the generated files into this directory instead of the default `~/.cache/trustify/<hash>/`. Useful when shipping a frozen schema alongside another tool.

## Loading a dataset

`trustify.load_dataset(path)` tokenizes the `.data` file and returns it as a pydantic model. The schema is generated (or reused from cache) automatically, so this single call is all you need.

Below is the dataset we will use, for reference:

In [ ]:
fNam = "upwind_simplified.data"
with open(fNam) as f:
    print(f.read())

And now we load it into the data model:

In [ ]:
ds = trustify.load_dataset(fNam)

`load_dataset` accepts:

- `filename` *(str | Path)* — the `.data` file to parse.
- `trust_root` *(str, optional)* and `projects` *(list[str], optional)* — same meaning as in `generate_schema`. Forwarded transparently when a schema needs to be generated.
- `schema` *(str | Path, optional)* — directory of a previously generated schema (the return value of `generate_schema`). When provided, `load_dataset` skips schema generation entirely and reuses that directory — useful in batch / scripted workflows where you want to be explicit about which schema you parse against.

Now **ds** contains our TRUST dataset as a Python object and we can finally play with it! We show several examples:

### Example: reading/adding a time scheme attribute

What is the current final time in our dataset ? Note in the below how the Python attributes exactly corresponds to the name of the attribute in the TRUST syntax.

In [ ]:
# Named object in the dataset are retrieved using the get() method:
ze_time_scheme = ds.get("sch")

# Then attributes can be read using the option name of the TRUST keyword, here 'tmax':
t_mx = ze_time_scheme.tmax
print("tmax is %g" % t_mx )

If we want to inspect the list of available attributes, we can do:

In [ ]:
print(ze_time_scheme.model_fields.keys())  # model_fields is provided by pydantic

Another (more complete) possibility is:

In [ ]:
from pprint import pprint  # Useful Python pretty print
pprint(ze_time_scheme.model_fields)

We can now set an extra parameter:

In [ ]:
ze_time_scheme.tinit = 0.0

Notice how this fail if you try to assign the wrong type (here a string instead of a double):

In [ ]:
# ze_time_scheme.tinit = "tutu"

### Example: reading the name of the first probe
Same story with a more complex things: the name of the first probe ...

In [ ]:
# Retrieve the problem named 'pb' in the dataset:
ze_pb = ds.get("pb")

# Read the first probe in the postprocessing block:
first_probe = ze_pb.post_processing.probes[0]
print(first_probe.nom_sonde)

... and then the x coordinate of the first point in the first probe: notice how some attributes might be painful and very dependent on how the TRUST grammar was built. In this case for example, the somewhat cryptic 'type' intermediate argument comes from the way the TRAD2 is defined.

In [ ]:
y = first_probe.type.points[0].pos[0]
print(y)

## Validating the dataset

On top of the automatic validation performed when you assign values, you can also explicitely request the model to validate itself by invoking:


In [ ]:
ds.self_validate()
# Or we can also validate only part of the dataset:
ze_time_scheme.self_validate()

This will fail (raising an exception) if any of the value contained is the model is not compliant with the type requested. This is particularly useful for complex types like lists, where pydantic can not perform the check when assigning the value.

## Writing back the dataset
The data can be modified ... and written back in a form of a new dataset!

Let's modify the y coordinate of the first probe:

In [ ]:
first_probe.type.points[0].pos[0] = -42.1  # as simple as that!

We can also for example delete the second probe:

In [ ]:
del(ze_pb.post_processing.probes[1])

Now we produce a new stream of tokens corresponding to this modified instance. This stream of tokens can then be simply joined to find back a textual dataset with the modifed data. Notice how all the other comments and formatting is kept unchanged.

In [ ]:
# Convert back the data into a stream of tokens:
newStream = ds.toDatasetTokens() 
# And write it out!
s = ''.join(ds.toDatasetTokens())
print(s)